# How to run this notebook

**What it does.** Takes a PlantVillage-style folder of leaf images, repairs the dataset
(removes duplicates and train/test leakage), rebuilds an honest 70/15/15 split, then
trains and evaluates **LeafGuardNet-BG** on it. Everything the report needs - metrics tables,
training curves, confusion matrices, per-class results - is produced by running top to
bottom.

### 1. Get the data

Either the full PlantVillage set (14 crops, 38 classes, 54,305 images):

```
git clone --filter=blob:none --sparse --depth 1 \
    https://github.com/spMohanty/PlantVillage-Dataset.git plantvillage_full
cd plantvillage_full && git sparse-checkout set raw/color
```

or the pre-split `data_split_38.zip` if it was shared with you.

### 2. Point `DATA_PATH` at it

The only line you must edit, in the config cell below. It accepts a **folder** or a
**.zip**. If the data is already split into `train/val/test`, those folders are pooled
and the split is rebuilt - that is deliberate, see "Why Part A exists" below.

- **Local:** `DATA_PATH = "plantvillage_full/raw/color"` (relative to this notebook)
- **Colab:** upload the zip to Drive, then
  `DATA_PATH = "/content/drive/MyDrive/data_split_38.zip"`, or set `UPLOAD_NOW = True`

### 3. Run all cells

On a GPU expect roughly 1.5-2 hours (Part A about 5 minutes, then 20 training epochs).
Without a GPU it is impractical - on Colab pick *Runtime > Change runtime type > GPU*.
Missing packages are pip-installed automatically by the config cell.

### 4. What you get

Everything lands in `crop-disease-38/`:

| File | What it is |
|---|---|
| `results_leafguard2.json` | every metric, plus the settings that produced them |
| `crop_disease_leafguard2.pth` | deployable weights + class order + preprocessing |
| `manifest_clean.csv` | the split, one row per image, with `group_key` |
| `log_leafguard2.csv`, `curves_*.png`, `confusion_*.png` | training history and figures |

**To train a different architecture**, change `MODEL_KEY` in the config cell - the
registry in Part C lists every valid key (`resnet50`, `mobilenet_v2`, `efficientnet_b0`,
`densenet121`, ...). Results files are named per model, so runs do not overwrite each
other and the Part D table compares whatever it finds.

---

> **What this model is for.** The other notebooks chase PlantVillage accuracy, where
> every model lands above 98 %. Tested on web images none of them had seen, the same
> models scored 27-47 % — they had learned leaf silhouette and a plain grey studio
> background, not lesion appearance (crop identified 13/15, disease only 8/15).
>
> LeafGuardNet-BG targets that gap rather than the leaderboard: a deliberately small
> backbone, CBAM attention so *where* the lesion sits is representable, heavy dropout,
> label smoothing, and augmentation aggressive enough that the studio look stops being
> a usable cue. Part E scores the unseen set in-notebook, so the honest number is
> reported next to the flattering one.
>
> Expect **lower** PlantVillage accuracy than DenseNet's 99.79 %. That is the trade
> being made on purpose.


> **What is different here.** Same architecture as `crop_disease_leafguard.ipynb`; the
> change is one transform. Every PlantVillage image is a leaf on a plain grey card, so
> "grey card + this silhouette" predicts the class without the model ever learning what
> a lesion looks like — which is why unseen accuracy sat at 27–47 % while PlantVillage
> accuracy was 99 %+. `RandomBackground` segments the leaf and composites it onto a
> random background 70 % of the time, so the background carries no information and the
> leaf is the only thing left to learn from.
>
> This is the strongest lever available *without new data*. It is still not a
> substitute for real field images: 15 unseen photos cannot validate it either way.


# Crop Disease Detection - Complete Pipeline (38 classes)

**One notebook, start to finish.** Point `DATA_PATH` at the PlantVillage folder and run
top to bottom. It repairs the dataset, rebuilds an honest split, trains the model and
writes every number and figure the report needs.

| | |
|---|---|
| **Part A** | Repairs the dataset: removes leakage, duplicates, mislabels, then rebuilds the split |
| **Part B** | Shared setup: seeding, transforms, datasets and loaders |
| **Part C** | Trains and evaluates `MODEL_KEY` (default SqueezeNet 1.1) |
| **Part D** | Comparison table, and inference on your own photos |

The custom architecture lives in its own notebook, **`lite_disease_net_v4_38.ipynb`**.
It reads the same `manifest_clean.csv` this notebook writes, so run Part A here first;
its results land in the same folder and appear automatically in the Part D table below.

### Why Part A exists

Splitting PlantVillage naively leaks: the same leaf, re-ingested under a new UUID or
photographed again seconds later, lands in both train and test. On the full 38-class
set that contaminates **7.5% of evaluation images**, worth about a point of free, fake
accuracy. Part A groups related images with union-find, keeps every group inside one
split, then re-runs the leakage scan as a pass/fail gate.

Nothing is deleted from your source folder. Every image gets a `status`, and only
`status == "ok"` images enter the split, so every exclusion stays auditable in
`excluded_images.csv`.

## 1. Config — the only cell anyone edits

Change `MODEL_KEY`, and set `DATA_PATH` to your raw dataset once. Valid model keys are listed in
Part B section 3; an invalid one fails immediately with the full list.

The banner this prints is your proof the run used the agreed settings. Screenshot it and paste it
next to your numbers.

In [1]:
# ============================ EDIT THESE ============================ #
MODEL_KEY  = "leafguard2"        # custom - background randomisation, see Part B

DATA_PATH  = "plantvillage_full/raw/color"   # .zip or folder; see the notes above
# Colab:  DATA_PATH = "/content/drive/MyDrive/data_split_38.zip"   (upload the zip to Drive)
# Or set UPLOAD_NOW = True to upload a zip straight into the session.
UPLOAD_NOW = False               # True = upload the zip here instead of reading Drive
BACKUP_DIR = None     # None to skip the Drive backup
# =================== DO NOT EDIT BELOW THIS LINE ==================== #

# ---- Part A: dataset repair (locked) ---- #
SEED     = 42
RATIOS   = {"train": 0.70, "val": 0.15, "test": 0.15}
HAMMING  = 4             # perceptual-hash distance counted as "near-identical"
MIN_EVAL = 30            # warn when a class has fewer eval images than this

# ---- Part B: training (locked) ---- #
EPOCHS, LR, BATCH_SIZE, IMG_SIZE = 20, 3e-4, 32, 224   # higher LR: new head, small backbone
LABEL_SMOOTHING = 0.1                        # honest confidences, see Part C
USE_CLASS_WEIGHTS = True
USE_AMP = True

import glob, hashlib, io, json, os, random, re, shutil, subprocess, sys, time, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules


def ensure(module, package=None):
    """Import module, pip-installing it once if missing. Keeps Colab and local identical."""
    try:
        return __import__(module)
    except ImportError:
        pkg = package or module
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        return __import__(module)


for _mod, _pkg in (("torch", None), ("torchvision", None), ("sklearn", "scikit-learn"),
                   ("matplotlib", None), ("seaborn", None), ("torchmetrics", None),
                   ("tqdm", None)):
    ensure(_mod, _pkg)

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torchvision
from PIL import Image, ImageFilter, ImageFile
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import (MulticlassF1Score, MulticlassPrecision,
                                         MulticlassRecall)
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False       # fail loudly on truncated files

# ---- constants, not knobs ---- #
EXPECTED_CLASSES = 38                        # full PlantVillage: 14 crops, 38 classes
TEAM_SIZE = 7
MEAN = [0.485, 0.456, 0.406]                 # ImageNet stats the pretrained weights expect
STD = [0.229, 0.224, 0.225]
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
SKIP_DIRS = {"__MACOSX", ".ipynb_checkpoints", ".git"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = Path("/content/crop-disease-38") if IN_COLAB else Path.cwd() / "crop-disease-38"
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "cache").mkdir(exist_ok=True)

# Where a previous run may have left outputs. First hit wins.
ARTIFACT_DIRS = [OUT_DIR, Path("/content/crop-disease-38"), Path("/content"),
                 Path("/content/drive/MyDrive/crop-disease-38"), Path("/content/drive/MyDrive"),
                 Path.cwd(), Path.cwd().parent]


def find_artifact(name):
    """First existing copy of an earlier output, or None."""
    for d in ARTIFACT_DIRS:
        try:
            p = d / name
            if p.is_file():
                return p
        except OSError:
            continue
    return None


random.seed(SEED)
np.random.seed(SEED)
assert abs(sum(RATIOS.values()) - 1.0) < 1e-9, "RATIOS must sum to 1.0"

RUN_STARTED = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
_gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "none"

LOCKED = [
    ("epochs", EPOCHS),
    ("learning rate", LR),
    ("batch size", BATCH_SIZE),
    ("image size", f"{IMG_SIZE}x{IMG_SIZE}"),
    ("seed", SEED),
    ("optimizer", f"Adam(lr={LR})"),
    ("scheduler", "ReduceLROnPlateau(max, factor=0.5, patience=3) on val macro-F1"),
    ("loss", "CrossEntropyLoss" + (" (class-weighted)" if USE_CLASS_WEIGHTS else "")),
    ("train augmentation", "RandomResizedCrop(0.8-1.0) + HFlip + VFlip + Rot(10)"),
    ("", "  + ColorJitter(brightness=0.2, contrast=0.2, saturation=0, hue=0)"),
    ("eval transform", f"Resize({round(IMG_SIZE * 8 / 7)}) + CenterCrop({IMG_SIZE})"),
    ("model selection", "best val macro-F1 (not accuracy)"),
    ("mixed precision", f"{USE_AMP} (active: {bool(USE_AMP and DEVICE.type == 'cuda')})"),
    ("determinism", "cudnn.deterministic=True, benchmark=False, seeded workers"),
    ("split", f"{RATIOS} grouped + stratified, seed {SEED}"),
    ("near-duplicate radius", f"dHash Hamming <= {HAMMING}"),
]

print("=" * 78)
print(f"  MODEL_KEY  ->  {MODEL_KEY}")
print("=" * 78)
print("  LOCKED FOR ALL 7 RUNS - screenshot this")
for _k, _v in LOCKED:
    print(f"    {_k:<22} {_v}")
print("-" * 78)
print(f"    {'device':<22} {DEVICE.type} ({_gpu})")
print(f"    {'torch':<22} {torch.__version__} / torchvision {torchvision.__version__}")
print(f"    {'output dir':<22} {OUT_DIR}")
print(f"    {'started':<22} {RUN_STARTED}")
print("=" * 78)
if DEVICE.type != "cuda":
    print("\nWARNING: no GPU. Runtime > Change runtime type > T4 GPU, or this takes hours.")

C:\Users\Welcome\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  MODEL_KEY  ->  leafguard2
  LOCKED FOR ALL 7 RUNS - screenshot this
    epochs                 20
    learning rate          0.0003
    batch size             32
    image size             224x224
    seed                   42
    optimizer              Adam(lr=0.0003)
    scheduler              ReduceLROnPlateau(max, factor=0.5, patience=3) on val macro-F1
    loss                   CrossEntropyLoss (class-weighted)
    train augmentation     RandomResizedCrop(0.8-1.0) + HFlip + VFlip + Rot(10)
                             + ColorJitter(brightness=0.2, contrast=0.2, saturation=0, hue=0)
    eval transform         Resize(256) + CenterCrop(224)
    model selection        best val macro-F1 (not accuracy)
    mixed precision        True (active: True)
    determinism            cudnn.deterministic=True, benchmark=False, seeded workers
    split                  {'train': 0.7, 'val': 0.15, 'test': 0.15} grouped + stratified, seed 42
    near-duplicate radius  dHash Hamming <= 4
---------

---
---
# PART A — Repair the dataset

Sections A1-A6 turn the raw download into a split you can trust. Output is
`manifest_clean.csv` plus `class_weights.json`, both used by Part B.

Nothing is deleted from your source folder. Every image gets a `status`, and only `status == "ok"`
images enter the split, so every exclusion stays auditable in `excluded_images.csv`.

## A1. Load the raw dataset

**Recommended:** zip your dataset folder, drag the zip into Google Drive in your browser, set
`DATA_PATH` to it. Drive survives runtime restarts so you upload once.

**Or:** set `UPLOAD_NOW = True` and upload here. Simpler, but slow for 330 MB and lost on restart.

`DATA_PATH` accepts a `.zip` or a folder. Folders that directly contain images become classes. If
your data is already split into `train/val/test`, all three are **pooled** — we are rebuilding the
split, so the old one is discarded (remembered only to measure how leaky it was). `.DS_Store` and
other non-image files are filtered out here.

In [2]:
SPLIT_TOKENS = {"train": "train", "training": "train", "val": "val", "valid": "val",
                "validation": "val", "test": "test", "testing": "test"}


DATA_HINTS = ("data", "plant", "village", "split", "leaf", "crop", "dataset")


def _looks_like_dataset(d):
    """True if d holds train/val/test, or enough class subfolders to be the data root."""
    try:
        subs = [x for x in d.iterdir() if x.is_dir()]
    except OSError:
        return False
    names = {x.name.lower() for x in subs}
    if {"train", "test"} <= names or {"train", "val"} <= names:
        return True
    with_images = 0
    for x in subs[:40]:
        try:
            if any(f.suffix.lower() in IMG_EXT for f in list(x.iterdir())[:20]):
                with_images += 1
        except OSError:
            continue
    return with_images >= 10


def discover_dataset(max_depth=3):
    """Look for a dataset zip or folder in the usual Colab places.

    Returns (zips, dirs). Depth-limited so a large Drive does not take minutes.
    """
    roots = [Path("/content"), Path("/content/drive/MyDrive"), Path.cwd()]
    zips, dirs, seen = [], [], set()
    for root in roots:
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            d, depth = stack.pop()
            key = str(d.resolve()) if d.exists() else str(d)
            if key in seen or depth > max_depth:
                continue
            seen.add(key)
            try:
                entries = list(d.iterdir())
            except OSError:
                continue
            for e in entries:
                if e.name in SKIP_DIRS or e.name.startswith("."):
                    continue
                if e.is_file() and e.suffix.lower() == ".zip":
                    zips.append(e)
                elif e.is_dir():
                    if _looks_like_dataset(e):
                        dirs.append(e)
                    else:
                        stack.append((e, depth + 1))
    hinted = [z for z in zips if any(h in z.name.lower() for h in DATA_HINTS)]
    return (hinted or zips), dirs


def _rank_candidates():
    """Best guesses first: a dataset-shaped folder, then a plausibly named zip."""
    zips, dirs = discover_dataset()
    return [(d, "folder") for d in dirs] + [(z, "zip") for z in zips]


def resolve_data(path_str):
    """Return a directory holding the dataset, extracting a zip if needed.

    If DATA_PATH does not exist, search Drive and /content rather than just failing -
    six other people run this notebook and the path is the one thing they must change.
    """
    p = Path(path_str)

    if not p.exists():
        print(f"DATA_PATH does not exist: {p}")
        print("searching Drive and /content for your dataset ...")
        found = _rank_candidates()

        if not found:
            raise FileNotFoundError(
                f"{p} not found, and no dataset zip or folder turned up under "
                f"/content/drive/MyDrive or /content.\n"
                f"Upload your data to Drive and set DATA_PATH to it, or set "
                f"UPLOAD_NOW = True to upload a zip directly.")

        if len(found) == 1:
            chosen, kind = found[0]
            print(f"\n  found exactly one candidate, using it:")
            print(f"    {chosen}   ({kind})")
            print(f"  to make this explicit, set:  DATA_PATH = {str(chosen)!r}\n")
            p = chosen
        else:
            print(f"\n  {len(found)} candidates found. Set DATA_PATH to the right one:\n")
            for c, kind in found[:12]:
                size = ""
                if kind == "zip":
                    try:
                        size = f"  ({c.stat().st_size / 1e6:.0f} MB)"
                    except OSError:
                        pass
                print(f"    DATA_PATH = {str(c)!r}{size}")
            raise FileNotFoundError(
                "DATA_PATH is ambiguous - copy one of the lines above into cell 1.")

    if p.is_dir():
        print(f"using folder: {p}")
        return p
    if p.suffix.lower() != ".zip":
        raise ValueError(f"DATA_PATH must be a folder or a .zip, got: {p}")
    dest = OUT_DIR / "raw"
    if dest.exists() and any(dest.rglob("*")):
        print(f"already extracted: {dest}")
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    print(f"extracting {p.name} ({p.stat().st_size / 1e6:.0f} MB) ...")
    with zipfile.ZipFile(p) as zf:
        zf.extractall(dest)
    return dest


def find_classes(root):
    """Return (class -> [paths], path -> original split or None)."""
    classes, orig, pooled, junk = defaultdict(list), {}, set(), 0
    for dirpath, _dirs, filenames in os.walk(root):
        d = Path(dirpath)
        imgs = [d / f for f in filenames if Path(f).suffix.lower() in IMG_EXT]
        junk += len(filenames) - len(imgs)
        if not imgs:
            continue
        if d.name.lower() in SPLIT_TOKENS:                 # images with no class folder
            print(f"  skipping {len(imgs)} images directly under {d}")
            continue
        if d.parent.name.lower() in SPLIT_TOKENS:
            s = SPLIT_TOKENS[d.parent.name.lower()]
            pooled.add(s)
            orig.update({str(p): s for p in imgs})
        classes[d.name].extend(imgs)

    if not classes:
        raise FileNotFoundError(f"no class folders with images under {root}")
    print(f"pooled existing splits {sorted(pooled)}, old split discarded on purpose"
          if pooled else "source is flat, no existing split")
    print(f"ignored {junk} non-image files (.DS_Store etc)")
    return {k: sorted(v) for k, v in sorted(classes.items())}, orig


# Mount Drive whenever we need it for input or backup.
if IN_COLAB and (str(DATA_PATH).startswith("/content/drive")
                 or (BACKUP_DIR and str(BACKUP_DIR).startswith("/content/drive"))):
    from google.colab import drive

    drive.mount("/content/drive")

if UPLOAD_NOW and IN_COLAB:
    from google.colab import files

    print("Select your zipped dataset ...")
    up = files.upload()
    name = next(iter(up))
    Path(name).write_bytes(up[name])
    SOURCE_DIR = resolve_data(name).resolve()
else:
    SOURCE_DIR = resolve_data(DATA_PATH).resolve()

DRIVE_OK = bool(BACKUP_DIR) and (not str(BACKUP_DIR).startswith("/content/drive")
                                 or Path("/content/drive/MyDrive").exists())
CLASSES, ORIG_SPLIT = find_classes(SOURCE_DIR)

print(f"\n{len(CLASSES)} classes, {sum(len(v) for v in CLASSES.values())} images\n")
for k, v in CLASSES.items():
    print(f"  {k:<50} {len(v):>6}")

using folder: plantvillage_full\raw\color


source is flat, no existing split
ignored 0 non-image files (.DS_Store etc)



38 classes, 54305 images

  Apple___Apple_scab                                    630
  Apple___Black_rot                                     621
  Apple___Cedar_apple_rust                              275
  Apple___healthy                                      1645
  Blueberry___healthy                                  1502
  Cherry_(including_sour)___Powdery_mildew             1052
  Cherry_(including_sour)___healthy                     854
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot    513
  Corn_(maize)___Common_rust_                          1192
  Corn_(maize)___Northern_Leaf_Blight                   985
  Corn_(maize)___healthy                               1162
  Grape___Black_rot                                    1180
  Grape___Esca_(Black_Measles)                         1383
  Grape___Leaf_blight_(Isariopsis_Leaf_Spot)           1076
  Grape___healthy                                       423
  Orange___Haunglongbing_(Citrus_greening)             5507
  Peach___Bac

## A2. Read every file

One pass per image, collecting four things: **SHA-256** (identical files), **dHash** (near-identical
files), whether it **actually decodes**, and size plus colour mode. Cached, so a runtime restart
does not repeat it.

In [3]:
UUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}___", re.I)
LEAF_DAY = re.compile(r"^(?P<sp>.*?\bLeaf\s*[\d.]+)\s*Day\s*[\d.]+\s*$", re.I)


def source_name(fn):
    """Original capture name, ingest UUID and extension stripped."""
    return UUID_RE.sub("", Path(fn).stem).strip()


def specimen_key(fn):
    """'GHLB_PS Leaf 39.1 Day 16' -> 'ghlb_ps leaf 39.1'. One physical leaf over time."""
    m = LEAF_DAY.match(source_name(fn))
    return m.group("sp").strip().lower() if m else None


def dhash(img, size=8):
    """64-bit perceptual hash: compare each pixel to its right neighbour."""
    px = list(img.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS).getdata())
    bits = pos = 0
    for r in range(size):
        b = r * (size + 1)
        for c in range(size):
            if px[b + c] < px[b + c + 1]:
                bits |= 1 << pos
            pos += 1
    return f"{bits:016x}"


def probe(path_str):
    rec = dict(path=path_str, error=None, bytes=None, sha256=None, dhash=None,
               img_format=None, mode=None, width=None, height=None)
    try:
        data = Path(path_str).read_bytes()
    except OSError as e:
        rec["error"] = f"unreadable: {e}"
        return rec
    rec["bytes"], rec["sha256"] = len(data), hashlib.sha256(data).hexdigest()
    if not data:
        rec["error"] = "zero-byte file"
        return rec
    try:                                          # structure check
        with Image.open(io.BytesIO(data)) as im:
            rec["img_format"], rec["mode"] = im.format, im.mode
            rec["width"], rec["height"] = im.size
            im.verify()
    except Exception as e:
        rec["error"] = f"verify failed: {type(e).__name__}: {e}"
        return rec
    try:                                          # full decode catches truncation
        with Image.open(io.BytesIO(data)) as im:
            im.load()
            rec["dhash"] = dhash(im)
    except Exception as e:
        rec["error"] = f"decode failed: {type(e).__name__}: {e}"
    return rec


df = pd.DataFrame([{"path": str(p), "filename": p.name, "class_raw": cls,
                    "source_name": source_name(p.name), "specimen": specimen_key(p.name),
                    "orig_split": ORIG_SPLIT.get(str(p))}
                   for cls, paths in CLASSES.items() for p in paths])
df["relpath"] = df["class_raw"] + "/" + df["filename"]
assert df["path"].is_unique

# Plain CSV cache: no pyarrow dependency, and a cache problem must never be fatal.
cache = OUT_DIR / "cache" / f"probe_{len(df)}.csv"
probed = None
if cache.exists():
    try:
        probed = pd.read_csv(cache, dtype={"sha256": str, "dhash": str})
        if len(probed) != len(df) or set(probed["path"]) != set(df["path"]):
            probed = None
            print("cache was written for different files, re-reading")
        else:
            print(f"loaded cache: {len(probed)} rows")
    except Exception as e:
        print(f"cache unreadable ({e}), re-reading")

if probed is None:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max(8, (os.cpu_count() or 2) * 4)) as ex:
        probed = pd.DataFrame(list(tqdm(ex.map(probe, df["path"]), total=len(df),
                                        desc="reading", unit="img")))
    print(f"read {len(probed)} files in {time.time() - t0:.0f}s")
    try:
        probed.to_csv(cache, index=False)
    except Exception as e:
        print(f"could not write cache ({e}) - harmless")

df = df.merge(probed, on="path", validate="one_to_one")
assert len(df), ("probe merge produced no rows: the cache holds different paths than "
                 f"the ones just scanned. Delete {OUT_DIR / 'cache'} and rerun.")
HAS_ORIG = df["orig_split"].notna().any()
print(f"decodable {int(df['error'].isna().sum())} | failed {int(df['error'].notna().sum())}")

loaded cache: 54305 rows
decodable 54305 | failed 0


## A3. Measure the leakage

`leakage_scan` counts how many eval images have a counterpart in train. It runs twice: here for the
baseline, and again in cell 7 as the pass/fail gate. Same ruler both times, which is the point.

In [4]:
def near_dup_pairs(d, hamming=HAMMING, bucket_cap=400):
    """Near-identical pairs. LSH banding: hashes within distance 4 must share one of
    4 disjoint 16-bit bands, which avoids an O(n^2) all-pairs scan."""
    have = d[d["dhash"].notna()]
    bands = defaultdict(list)
    for idx, h in zip(have.index, have["dhash"]):
        for b in range(4):
            bands[(b, h[b * 4:(b + 1) * 4])].append(idx)

    hashes, seen, pairs = have["dhash"].to_dict(), set(), []
    for bucket in bands.values():
        if len(bucket) < 2 or len(bucket) > bucket_cap:
            continue
        for i in range(len(bucket)):
            for j in range(i + 1, len(bucket)):
                a, b = sorted((bucket[i], bucket[j]))
                if (a, b) in seen:
                    continue
                seen.add((a, b))
                if bin(int(hashes[a], 16) ^ int(hashes[b], 16)).count("1") <= hamming:
                    pairs.append((a, b))
    return pairs


def leakage_scan(d, split_col, pairs=None):
    """Cross-split overlap by every signal. All zeros is the only acceptable result."""
    d = d[d[split_col].notna()]
    g_hash = {k: v for k, v in d.groupby("sha256").groups.items() if len(v) > 1}
    g_src = {k: v for k, v in d.groupby("source_name").groups.items() if len(v) > 1}
    spec = d[d["specimen"].notna()]
    g_spec = {k: v for k, v in spec.groupby("specimen").groups.items() if len(v) > 1}
    if pairs is None:
        pairs = near_dup_pairs(d)
    live_pairs = [(a, b) for a, b in pairs if a in d.index and b in d.index]

    n_cross = lambda gs: sum(1 for g in gs if d.loc[g, split_col].nunique() > 1)
    out = {
        "identical_files_across_splits": n_cross(g_hash.values()),
        "same_capture_across_splits": n_cross(g_src.values()),
        "same_leaf_across_splits": n_cross(g_spec.values()),
        "near_identical_pairs_across_splits": sum(
            1 for a, b in live_pairs if d.at[a, split_col] != d.at[b, split_col]),
    }

    tainted = set()
    for groups in (g_hash, g_src, g_spec):
        for g in groups.values():
            if d.loc[g, split_col].nunique() > 1:
                tainted.update(i for i in g if d.at[i, split_col] != "train")
    for a, b in live_pairs:
        if d.at[a, split_col] != d.at[b, split_col]:
            tainted.update(i for i in (a, b) if d.at[i, split_col] != "train")

    n_eval = int((d[split_col] != "train").sum())
    out["contaminated_eval_images"] = len(tainted)
    out["eval_images"] = n_eval
    out["contamination_pct"] = round(100 * len(tainted) / n_eval, 3) if n_eval else 0.0
    return out


def naive_split(d):
    """The split most people write: stratify by class, shuffle, slice. Ignores groups.
    Only used to show the baseline when your data has no existing split."""
    out = pd.Series(index=d.index, dtype="object")
    for _c, sub in d.groupby("class_raw"):
        idx = sub.sample(frac=1.0, random_state=SEED).index
        a = int(round(len(idx) * RATIOS["train"]))
        b = a + int(round(len(idx) * RATIOS["val"]))
        out.loc[idx[:a]], out.loc[idx[a:b]], out.loc[idx[b:]] = "train", "val", "test"
    return out


all_pairs = near_dup_pairs(df)
counts = df["class_raw"].value_counts()

if HAS_ORIG:
    baseline, baseline_kind = leakage_scan(df, "orig_split", all_pairs), "your existing split"
else:
    df["naive_split"] = naive_split(df)
    baseline = leakage_scan(df, "naive_split", all_pairs)
    baseline_kind = "a naive stratified split (what you would have built)"

print(f"images          {len(df)}")
print(f"classes         {df['class_raw'].nunique()}")
print(f"unreadable      {int(df['error'].notna().sum())}")
print(f"identical files {len(df) - df['sha256'].nunique()}")
print(f"near-identical  {len(all_pairs)} pairs")
print(f"same-leaf sets  {df['specimen'].nunique()}")
print(f"imbalance       {counts.max() / counts.min():.0f}x  "
      f"({counts.idxmax()} {counts.max()} vs {counts.idxmin()} {counts.min()})")
print(f"colour modes    {df['mode'].value_counts().to_dict()}")
print(f"\nLEAKAGE in {baseline_kind}:")
print(json.dumps(baseline, indent=2))
print(f"\n>>> {baseline['contaminated_eval_images']} of {baseline['eval_images']} eval images "
      f"({baseline['contamination_pct']}%) have a twin in train")

images          54305
classes         38
unreadable      0
identical files 21
near-identical  259 pairs
same-leaf sets  144
imbalance       36x  (Orange___Haunglongbing_(Citrus_greening) 5507 vs Potato___healthy 152)
colour modes    {'RGB': 54304, 'RGBA': 1}

LEAKAGE in a naive stratified split (what you would have built):
{
  "identical_files_across_splits": 11,
  "same_capture_across_splits": 985,
  "same_leaf_across_splits": 22,
  "near_identical_pairs_across_splits": 113,
  "contaminated_eval_images": 1228,
  "eval_images": 16293,
  "contamination_pct": 7.537
}

>>> 1228 of 16293 eval images (7.537%) have a twin in train


## A4. Clean up and label

Three things at once:

1. Drop **unreadable** files and **identical duplicates** (keep one copy of each).
2. Split class names into **crop** and **disease**. PlantVillage delimiters are inconsistent
   (`Potato___Early_blight` vs `Tomato_Late_blight`), so we try the triple underscore first, then
   fall back to a known crop prefix.
3. Flag **mislabelled** files using the capture code in the filename: `GH_HL` / `JR_HL` mean
   *healthy leaf*, `GHLB` means *late blight*. A healthy code inside a disease class means the label
   is wrong. On your data this catches two clean green leaves in `Tomato_Late_blight`.

Nothing is deleted from your source folder. Rows get a `status` and only `ok` rows get split, so
every exclusion stays auditable.

In [5]:
df["status"], df["status_reason"] = "ok", ""


def exclude(mask, status, reason):
    hit = mask.fillna(False) & (df["status"] == "ok")
    df.loc[hit, ["status", "status_reason"]] = [status, reason]
    print(f"{status:<12} {int(hit.sum()):>5}  {reason}")
    return int(hit.sum())


exclude(df["error"].notna(), "quarantined", "unreadable or corrupt")
rank = (df[df["status"] == "ok"].sort_values("relpath", kind="mergesort")
        .groupby("sha256").cumcount().reindex(df.index))
exclude(rank > 0, "dropped", "identical duplicate of another file")
assert df.loc[df["status"] == "ok", "sha256"].is_unique, "duplicates survived"

# ---- crop / disease ---- #
KNOWN_CROPS = ["Pepper__bell", "Pepper,_bell", "Pepper_bell", "Potato", "Tomato", "Apple",
               "Blueberry", "Cherry_(including_sour)", "Cherry", "Corn_(maize)", "Corn",
               "Grape", "Orange", "Peach", "Raspberry", "Soybean", "Squash", "Strawberry"]


def parse_class(raw):
    if "___" in raw:
        crop, disease = raw.split("___", 1)
    else:
        for c in sorted(KNOWN_CROPS, key=len, reverse=True):
            if raw.startswith(c):
                crop, disease = c, raw[len(c):]
                break
        else:
            crop, disease = raw, "unknown"
    tidy = lambda s: re.sub(r"\s+", " ", s.replace("_", " ")).strip(" ,-")
    return tidy(crop) or "unknown", tidy(disease) or "unknown"


parsed = df["class_raw"].map(parse_class)
df["crop"] = [p[0] for p in parsed]
df["disease"] = [p[1] for p in parsed]
df["is_healthy"] = df["disease"].str.lower().str.contains("healthy")

taxonomy = (df[["class_raw", "crop", "disease", "is_healthy"]].drop_duplicates()
            .sort_values(["crop", "disease"]).reset_index(drop=True))
taxonomy["class_id"] = range(len(taxonomy))
bad = taxonomy.loc[taxonomy["crop"] == "unknown", "class_raw"].tolist()
assert not bad, f"could not parse, add the crop to KNOWN_CROPS: {bad}"
df = df.merge(taxonomy[["class_raw", "class_id"]], on="class_raw")

# ---- mislabelled files ---- #
healthy_code = df["source_name"].str.match(r"^(?:GH|JR|RS)_HL\b", case=False).fillna(False)
disease_code = df["source_name"].str.contains(
    r"(B\.Spot|Bact\.Sp|GHLB|YLCV|Early|Late|Mold|Septoria)", case=False).fillna(False)
conflict = (healthy_code & ~df["is_healthy"]) | (disease_code & df["is_healthy"])
n_conflict = int(conflict.sum())

print(f"\n{len(taxonomy)} classes across {taxonomy['crop'].nunique()} crops")
print(taxonomy[["class_id", "crop", "disease"]].to_string(index=False))
print(f"\nmislabelled files: {n_conflict}")
if n_conflict:
    print(df.loc[conflict, ["relpath", "source_name", "disease"]].to_string(index=False))
    exclude(conflict, "quarantined", "label contradicts the filename capture code")

print(f"\nusable: {int((df['status'] == 'ok').sum())} / {len(df)}")

quarantined      0  unreadable or corrupt
dropped         21  identical duplicate of another file

38 classes across 14 crops
 class_id                    crop                              disease
        0                   Apple                           Apple scab
        1                   Apple                            Black rot
        2                   Apple                     Cedar apple rust
        3                   Apple                              healthy
        4               Blueberry                              healthy
        5 Cherry (including sour)                       Powdery mildew
        6 Cherry (including sour)                              healthy
        7            Corn (maize)  Cercospora leaf spot Gray leaf spot
        8            Corn (maize)                          Common rust
        9            Corn (maize)                 Northern Leaf Blight
       10            Corn (maize)                              healthy
       11             

C:\Users\Welcome\AppData\Local\Temp\ipykernel_23916\1082494420.py:51: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  disease_code = df["source_name"].str.contains(


## A5. Group related images

This is the actual fix. Three signals say "not independent samples", merged with union-find so
grouping is **transitive** — if A relates to B and B to C, all three land in one group.

| Signal | Catches |
|---|---|
| dHash within distance 4 | Consecutive shots of the same leaf |
| Same specimen (`Leaf 39 Day 16`) | One leaf photographed across days |
| Same source capture name | The same photo re-ingested under a new UUID |

In [6]:
class UnionFind:
    def __init__(self, items):
        self.p = {i: i for i in items}

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        lo, hi = sorted((ra, rb))
        self.p[hi] = lo
        return True


live = df[df["status"] == "ok"]
uf = UnionFind(live.index.tolist())
edges = Counter()

for a, b in near_dup_pairs(live):                                    # signal 1
    edges["near_identical"] += uf.union(a, b)
for col, name in (("specimen", "same_leaf"), ("source_name", "same_capture")):   # 2 and 3
    sub = live[live[col].notna()]
    for _k, idx in sub.groupby(col).groups.items():
        for other in idx[1:]:
            edges[name] += uf.union(idx[0], other)

df["group_root"] = pd.Series({i: uf.find(i) for i in live.index})
order = (df[df["status"] == "ok"].groupby("group_root")["relpath"].min()
         .sort_values(kind="mergesort"))
df["group_key"] = df["group_root"].map({r: f"g{n:06d}" for n, r in enumerate(order.index)})

live = df[df["status"] == "ok"]
sizes = live["group_key"].value_counts()
multi = sizes[sizes > 1]

# One label per group by majority vote; ties alphabetical to stay deterministic.
primary = (live.groupby(["group_key", "class_raw"]).size().reset_index(name="n")
           .sort_values(["group_key", "n", "class_raw"], ascending=[True, False, True],
                        kind="mergesort")
           .groupby("group_key").first()["class_raw"].rename("primary_class"))
groups = live.groupby("group_key").size().rename("n_images").to_frame().join(primary).reset_index()
assert groups["n_images"].sum() == len(live)

print(f"merges by signal   {dict(edges)}")
print(f"groups             {len(sizes)} for {len(live)} images")
print(f"multi-image groups {len(multi)} covering {int(multi.sum())} images")
print(f"largest group      {int(sizes.max())} images")
print(f"\nThose {int(multi.sum())} images are what a normal split would have torn apart.")

merges by signal   {'near_identical': 232, 'same_leaf': 68, 'same_capture': 2192}
groups             51790 for 54282 images
multi-image groups 2355 covering 4847 images
largest group      7 images

Those 4847 images are what a normal split would have torn apart.


## A6. Rebuild the split and prove it is clean

Every group stays in one split, and every class keeps its 70/15/15. We walk each class's groups
largest-first and give each to whichever split is furthest below target — largest-first matters,
because a big group placed last can overshoot with no way to undo it.

The gate re-runs the leakage scan and raises if anything is non-zero. It should pass by
construction, which is exactly why running it is worth it: it tests the grouping code, not just the
data.

In [7]:
def grouped_split(groups):
    """Assign each group to one split, keeping each class near its target ratio."""
    order = {s: i for i, s in enumerate(RATIOS)}
    out = {}
    for cls, sub in groups.groupby("primary_class", sort=True):
        sub = (sub.sample(frac=1.0, random_state=SEED)
               .sort_values("n_images", ascending=False, kind="mergesort"))
        total = int(sub["n_images"].sum())
        target = {s: total * r for s, r in RATIOS.items()}
        got = dict.fromkeys(RATIOS, 0)
        if len(sub) < len(RATIOS):
            print(f"  warning: {cls} has only {len(sub)} group(s)")
        for gk, n in zip(sub["group_key"], sub["n_images"]):
            best = min(RATIOS, key=lambda s: (-(target[s] - got[s]), order[s]))
            out[gk] = best
            got[best] += int(n)
    return pd.Series(out, name="split")


df["split"] = df["group_key"].map(grouped_split(groups))
df.loc[df["status"] != "ok", "split"] = None
clean = df[df["status"] == "ok"].copy()
after = leakage_scan(clean, "split")

# ------------------------------- the gate ------------------------------- #
fails = [f"{k} = {v}, expected 0" for k, v in after.items()
         if (k.endswith("across_splits") or k == "contaminated_eval_images") and v != 0]
spanning = clean.groupby("group_key")["split"].nunique()
if (spanning > 1).any():
    fails.append(f"{int((spanning > 1).sum())} groups span more than one split")
if clean["split"].isna().any():
    fails.append(f"{int(clean['split'].isna().sum())} usable images have no split")
for s in RATIOS:
    got = float((clean["split"] == s).mean())
    if abs(got - RATIOS[s]) > 0.02:
        fails.append(f"{s} is {got:.2%}, target {RATIOS[s]:.0%}")
missing = [(c, s) for c in clean["class_raw"].unique() for s in RATIOS
           if not ((clean["class_raw"] == c) & (clean["split"] == s)).any()]
if missing:
    fails.append(f"class/split combos with no images: {missing[:5]}")

print(json.dumps(after, indent=2))
if fails:
    raise AssertionError("DO NOT TRAIN ON THIS:\n  - " + "\n  - ".join(fails))

removed = len(df) - len(clean)
print("\n" + "=" * 62)
print("PASSED - no train/eval overlap remains")
print("=" * 62)
print(pd.DataFrame([
    {"problem": "eval images with a twin in train",
     "before": f"{baseline['contaminated_eval_images']} ({baseline['contamination_pct']}%)",
     "after": f"{after['contaminated_eval_images']} ({after['contamination_pct']}%)"},
    {"problem": "identical files across splits",
     "before": baseline["identical_files_across_splits"],
     "after": after["identical_files_across_splits"]},
    {"problem": "near-identical pairs across splits",
     "before": baseline["near_identical_pairs_across_splits"],
     "after": after["near_identical_pairs_across_splits"]},
    {"problem": "same leaf across splits",
     "before": baseline["same_leaf_across_splits"], "after": after["same_leaf_across_splits"]},
    {"problem": "unreadable files", "before": int(df["error"].notna().sum()), "after": 0},
    {"problem": "mislabelled files", "before": n_conflict, "after": 0},
]).to_string(index=False))

tbl = pd.crosstab(clean["class_raw"], clean["split"]).reindex(columns=list(RATIOS), fill_value=0)
xt = (pd.crosstab(clean["class_raw"], clean["split"], normalize="index")
      .reindex(columns=list(RATIOS), fill_value=0.0))
dev = (xt - pd.Series(RATIOS)).abs().max(axis=1)

print(f"\nbaseline measured on: {baseline_kind}")
print(f"cost: {removed} images removed ({100 * removed / len(df):.2f}%)")
for s in RATIOS:
    n = int((clean["split"] == s).sum())
    print(f"  {s:<6} {n:>6}  ({n / len(clean):.2%})")
print(f"largest per-class drift from 70/15/15: {dev.max():.2%} ({dev.idxmax()})")

thin = [{"class": c, "split": s, "n": int(tbl.at[c, s]),
         "recall_95%_CI": f"+/-{1.96 * (0.5 / max(int(tbl.at[c, s]), 1) ** 0.5):.0%}"}
        for c in tbl.index for s in ("val", "test") if int(tbl.at[c, s]) < MIN_EVAL]
if thin:
    print(f"\nWARNING - too few eval images for a stable per-class metric:")
    print(pd.DataFrame(thin).to_string(index=False))
    print("Quote confidence intervals for these, or use k-fold CV.")

{
  "identical_files_across_splits": 0,
  "same_capture_across_splits": 0,
  "same_leaf_across_splits": 0,
  "near_identical_pairs_across_splits": 0,
  "contaminated_eval_images": 0,
  "eval_images": 16285,
  "contamination_pct": 0.0
}

PASSED - no train/eval overlap remains
                           problem        before    after
  eval images with a twin in train 1228 (7.537%) 0 (0.0%)
     identical files across splits            11        0
near-identical pairs across splits           113        0
           same leaf across splits            22        0
                  unreadable files             0        0
                 mislabelled files             2        0

baseline measured on: a naive stratified split (what you would have built)
cost: 23 images removed (0.04%)
  train   37997  (70.00%)
  val      8149  (15.01%)
  test     8136  (14.99%)
largest per-class drift from 70/15/15: 15.96% (Strawberry___healthy)

WARNING - too few eval images for a stable per-class metric:
 

## A7. Save the manifest

`manifest_clean.csv` is what step 2 reads: one row per usable image with its split, labels and
`group_key`.

**Use the `split` column as-is.** If you later want cross-validation folds, split on `group_key`,
or you reintroduce exactly the leakage this notebook removed.

In [8]:
COLS = ["relpath", "path", "filename", "split", "class_raw", "class_id", "crop", "disease",
        "is_healthy", "group_key", "sha256", "dhash", "source_name", "specimen",
        "width", "height", "mode", "img_format", "bytes", "status", "status_reason", "error"]
manifest = (df.reindex(columns=COLS)
            .sort_values(["split", "class_raw", "relpath"], na_position="last")
            .reset_index(drop=True))

# Class weights for a weighted loss, from the TRAIN split only. Using all the data
# here would leak the eval distribution into training.
tr = clean.loc[clean["split"] == "train", "class_raw"]
freq = tr.value_counts()
weights = {str(c): round(len(tr) / (len(freq) * int(n)), 6) for c, n in freq.items()}

manifest.to_csv(OUT_DIR / "manifest.csv", index=False)
manifest[manifest["status"] == "ok"].to_csv(OUT_DIR / "manifest_clean.csv", index=False)
manifest[manifest["status"] != "ok"][
    ["relpath", "class_raw", "status", "status_reason", "error"]].to_csv(
    OUT_DIR / "excluded_images.csv", index=False)
taxonomy.to_csv(OUT_DIR / "taxonomy.csv", index=False)
(OUT_DIR / "class_weights.json").write_text(json.dumps(weights, indent=2))
(OUT_DIR / "fix_report.json").write_text(json.dumps({
    "seed": SEED, "ratios": RATIOS, "hamming": HAMMING, "source": str(SOURCE_DIR),
    "baseline_kind": baseline_kind, "images_in": len(df), "images_usable": len(clean),
    "images_removed": removed, "classes": int(clean["class_raw"].nunique()),
    "crops": int(clean["crop"].nunique()), "groups": int(clean["group_key"].nunique()),
    "split_counts": {str(k): int(v) for k, v in clean["split"].value_counts().items()},
    "leakage_before": baseline, "leakage_after": after, "verdict": "PASSED",
}, indent=2, default=str))

if DRIVE_OK:
    dest = Path(BACKUP_DIR)
    dest.mkdir(parents=True, exist_ok=True)
    for f in OUT_DIR.glob("*.*"):
        shutil.copy2(f, dest / f.name)
    print(f"backed up to {dest}")

for f in sorted(OUT_DIR.glob("*.*")):
    print(f"  {f.name:<22} {f.stat().st_size / 1024:>9.1f} KB")

print(f"""
DATASET READY   {len(clean)} usable of {len(df)}   ({removed} removed)
  {clean['class_raw'].nunique()} classes across {clean['crop'].nunique()} crops
  train/val/test  {int((clean['split'] == 'train').sum())} / """
      f"""{int((clean['split'] == 'val').sum())} / {int((clean['split'] == 'test').sum())}
  train/eval overlap  {baseline['contamination_pct']}%  ->  {after['contamination_pct']}%
  seed {SEED}, re-running gives the identical split
  main file  {OUT_DIR / 'manifest_clean.csv'}

STEP 2 NOTES
  Use the `split` column as-is. Do not re-split.
  Apply class_weights.json - imbalance is """
      f"""{counts.max() / counts.min():.0f}x, so accuracy will look
    good while the rare diseases quietly fail. Report macro-F1 and per-class recall.
  Call .convert("RGB") on every image; colour modes are not uniform.
  These are lab photos of single leaves on plain backgrounds, so real field photos
    will score lower. Keep a few real phone pictures aside to sanity-check.
""")

  best_alexnet.pth        223285.8 KB
  best_densenet121.pth     27912.9 KB
  best_googlenet.pth       22203.7 KB
  best_leafguard.pth        6474.3 KB
  best_lenet5.pth            255.9 KB
  best_resnet18.pth        43812.5 KB
  best_squeezenet1_1.pth    2918.6 KB
  best_vgg16.pth          525075.2 KB
  bg_randomisation_example.png     111.8 KB
  class_weights.json           1.6 KB
  confusion_test_alexnet.png     346.4 KB
  confusion_test_densenet121.png     343.5 KB
  confusion_test_googlenet.png     343.7 KB
  confusion_test_leafguard.png     343.7 KB
  confusion_test_lenet5.png     364.9 KB
  confusion_test_resnet18.png     344.0 KB
  confusion_test_squeezenet1_1.png     346.4 KB
  confusion_test_vgg16.png     345.4 KB
  confusion_val_alexnet.png     345.6 KB
  confusion_val_densenet121.png     343.5 KB
  confusion_val_googlenet.png     343.9 KB
  confusion_val_leafguard.png     343.4 KB
  confusion_val_lenet5.png     366.1 KB
  confusion_val_resnet18.png     343.4 KB
  confusion_

---
---
# PART B - Shared setup

Part A left the clean, split, labelled dataset in the `clean` DataFrame. Everything
below is shared by **both** models, so they train on identical data, identical
transforms and identical class ordering - the only way the comparison in Part D means
anything.

## B1. Reproducibility

Every reachable source of randomness gets the same seed, and cuDNN is put in deterministic mode.

This matters more than it looks when comparing architectures. Two runs of the *same* architecture
differ by a few tenths of a point from weight-init luck alone, which is the same margin we will be
ranking models on. Without this, "MobileNet beat SqueezeNet" is not a claim we could defend.

In [9]:
def seed_everything(seed):
    """Every source of randomness we can reach, in one place."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # no-op without a GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """DataLoader workers get their own seeded RNGs, derived from the loader generator."""
    ws = torch.initial_seed() % (2 ** 32)
    np.random.seed(ws)
    random.seed(ws)


seed_everything(SEED)
GEN = torch.Generator()
GEN.manual_seed(SEED)
RNG = np.random.default_rng(SEED)             # for figure sampling only

print(f"seeded everything with {SEED}")
print(f"  cudnn.deterministic {torch.backends.cudnn.deterministic}")
print(f"  cudnn.benchmark     {torch.backends.cudnn.benchmark}")
print(f"  PYTHONHASHSEED      {os.environ['PYTHONHASHSEED']}")

seeded everything with 42
  cudnn.deterministic True
  cudnn.benchmark     False
  PYTHONHASHSEED      42


## B2. Datasets and loaders

We read straight from the `clean` DataFrame rather than copying files into
`data_clean/train/<class>/...` for `ImageFolder`. Materialising the tree would mean copying ~20,600
small files on every run, for every team member, and doubling Colab disk for no benefit.

`ManifestDataset` exposes `.samples` and `.classes` so it is a drop-in for `ImageFolder`
everywhere downstream.

**Class order matters for comparability.** `CLASS_NAMES = sorted(unique class_raw)` reproduces
exactly the order `ImageFolder` would have produced from directory names, so class indices match
any run done the old way.

**Augmentation note:** flips and rotation are free here because leaves have no canonical
orientation. Saturation and hue jitter are deliberately **0** — shifting hue turns healthy green
toward chlorotic yellow, and that is a real symptom in several of these classes, so it would teach
the model to ignore the very signal we want.

In [10]:
# Part A produced this dataset in-session, so it is clean by construction.
DATA_SOURCE = "clean (repaired in Part A of this notebook)"
LEAKAGE_FIXED = True

CLASS_NAMES = sorted(clean["class_raw"].unique())      # == ImageFolder ordering
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
assert N_CLASSES == EXPECTED_CLASSES, f"{N_CLASSES} classes, expected {EXPECTED_CLASSES}"

# ---- resolve image paths ---------------------------------------------------- #
# `path` is absolute and written this session, so it normally just works. The fallback
# covers reruns where the data was re-extracted somewhere else.
_probe = clean["path"].head(50)
if not all(Path(p).is_file() for p in _probe):
    print("stored paths do not resolve - rebuilding a filename index from disk")
    index = {}
    for dirpath, dirnames, filenames in os.walk(SOURCE_DIR):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for f in filenames:
            if Path(f).suffix.lower() in IMG_EXT:
                index.setdefault(f, str(Path(dirpath) / f))
    missing = [f for f in clean["filename"] if f not in index]
    assert not missing, f"{len(missing)} images not found on disk, e.g. {missing[:3]}"
    clean = clean.assign(path=[index[f] for f in clean["filename"]])
    print(f"remapped {len(clean)} paths")
assert clean["path"].map(lambda p: Path(p).is_file()).all(), "some image paths do not exist"

# ---- transforms ------------------------------------------------------------- #
RESIZE = round(IMG_SIZE * 8 / 7)                       # 256 for 224, the standard ratio

# Domain-robust augmentation. PlantVillage is one leaf, centred, on plain grey card
# under even light; the images this has to survive are none of those things. Each
# transform below removes one crutch the model would otherwise lean on:
#
#   scale=(0.4, 1.0)   a 0.8-1.0 crop always shows the whole leaf, so silhouette alone
#                      identifies the crop. Cropping to 40% forces lesion texture.
#   rotation 30 + perspective   field photos are not shot flat-on from above.
#   saturation/hue     camera and daylight vary; hue stays small (0.05) because a large
#                      hue shift turns a healthy leaf yellow and corrupts the label.
#   grayscale p=0.1    occasionally removes colour entirely, so texture must carry it.
#   blur               web images vary in focus and resolution.
#   RandomErasing      occlusion: overlapping leaves, hands, shadows.

# ---- background randomisation: the transform that attacks the real shortcut -- #
class RandomBackground:
    """Segment the leaf and paste it onto a random background.

    Why this and not more blur/crop: measured on unseen photos, the models got the
    crop right 13/15 but the disease only 8/15. They leaned on leaf silhouette and
    the plain studio card. Ordinary augmentation only interpolates inside the domain
    it is given, so it cannot remove a cue that is present in every training image.
    Replacing the background makes that cue carry no information at all.

    The mask is a cheap colour rule rather than a segmentation network: PlantVillage
    backgrounds are near-neutral grey/black while leaves are green or brown, so
    excess-green plus saturation separates them (verified across classes). A rough
    mask is fine here - the goal is to destroy the background cue, not to produce a
    clean cut-out.
    """

    def __init__(self, p=0.7):
        self.p = p

    @staticmethod
    def _mask(a):
        r, g, b = a[..., 0], a[..., 1], a[..., 2]
        mx, mn = a.max(2), a.min(2)
        sat = np.where(mx > 0, (mx - mn) / np.maximum(mx, 1), 0)
        exg = 2 * g - r - b
        return ((sat > 0.18) | (exg > 18)) & (mx > 25)

    @staticmethod
    def _background(h, w, rng):
        """Solid, noise, gradient or coarse blobs - crude stand-ins for soil, sky,
        mulch and out-of-focus foliage. Variety matters more than realism: the point
        is that no single background statistic predicts the class."""
        kind = rng.integers(4)
        if kind == 0:                                   # solid colour
            return np.full((h, w, 3), rng.integers(20, 200, 3), dtype=np.float32)
        if kind == 1:                                   # fine noise
            base = rng.integers(20, 180, 3).astype(np.float32)
            return np.clip(base + rng.normal(0, 40, (h, w, 3)), 0, 255).astype(np.float32)
        if kind == 2:                                   # smooth gradient
            c1, c2 = rng.integers(0, 255, 3), rng.integers(0, 255, 3)
            t = np.linspace(0, 1, w if rng.random() < 0.5 else h, dtype=np.float32)
            ramp = (c1[None, :] * (1 - t[:, None]) + c2[None, :] * t[:, None])
            ramp = ramp[None, :, :] if ramp.shape[0] == w else ramp[:, None, :]
            return np.broadcast_to(ramp, (h, w, 3)).astype(np.float32).copy()
        small = rng.integers(0, 255, (max(h // 16, 2), max(w // 16, 2), 3)).astype(np.uint8)
        return np.asarray(Image.fromarray(small).resize((w, h), Image.BILINEAR),
                          dtype=np.float32)

    def __call__(self, img):
        if random.random() > self.p:
            return img
        a = np.asarray(img.convert("RGB"), dtype=np.float32)
        m = self._mask(a)
        frac = m.mean()
        if not (0.05 < frac < 0.97):        # mask looks wrong - leave the image alone
            return img
        rng = np.random.default_rng(random.randrange(2 ** 31))
        h, w = a.shape[:2]
        bg = self._background(h, w, rng)
        alpha = m.astype(np.float32)[..., None]
        # feather the edge so the model cannot key on a hard cut-out boundary
        alpha = np.asarray(Image.fromarray((alpha[..., 0] * 255).astype(np.uint8))
                           .filter(ImageFilter.GaussianBlur(1.5)),
                           dtype=np.float32)[..., None] / 255.0
        return Image.fromarray(np.clip(a * alpha + bg * (1 - alpha), 0, 255)
                               .astype(np.uint8))


train_tf = transforms.Compose([
    RandomBackground(p=0.7),          # <- runs first, on the full-frame image
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.4, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
eval_tf = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class ManifestDataset(Dataset):
    """Reads images listed in the manifest. Drop-in compatible with ImageFolder."""

    def __init__(self, frame, transform):
        self.samples = [(r.path, CLASS_TO_IDX[r.class_raw])
                        for r in frame.itertuples(index=False)]
        self.classes = list(CLASS_NAMES)
        self.class_to_idx = dict(CLASS_TO_IDX)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as im:
            img = im.convert("RGB")        # one source image is RGBA; without this it crashes
        return self.transform(img), label


parts = {s: clean[clean["split"] == s].reset_index(drop=True) for s in RATIOS}

train_set = ManifestDataset(parts["train"], train_tf)
train_eval_set = ManifestDataset(parts["train"], eval_tf)   # no aug, comparable to val/test
val_set = ManifestDataset(parts["val"], eval_tf)
test_set = ManifestDataset(parts["test"], eval_tf)

# Windows: spawned DataLoader workers cannot import classes defined in a notebook
NUM_WORKERS = 0 if os.name == "nt" else (2 if DEVICE.type == "cuda" else 0)
PIN = DEVICE.type == "cuda"


def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=PIN, drop_last=False, generator=GEN,
                      worker_init_fn=seed_worker,
                      persistent_workers=(NUM_WORKERS > 0))


train_loader = make_loader(train_set, True)
train_eval_loader = make_loader(train_eval_set, False)
val_loader = make_loader(val_set, False)
test_loader = make_loader(test_set, False)

# ---- report -------------------------------------------------------------- #
tab = (pd.crosstab(clean["class_raw"], clean["split"])
       .reindex(index=CLASS_NAMES, columns=list(RATIOS), fill_value=0))
tr = tab["train"]

print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)}   "
      f"batch {BATCH_SIZE}, workers {NUM_WORKERS}")
print(f"imbalance {tr.max() / tr.min():.0f}x in train "
      f"({tr.idxmax()} {tr.max()} vs {tr.idxmin()} {tr.min()})\n")
print(f"{'idx':<4} {'class':<46} {'train':>6} {'val':>5} {'test':>5}")
for i, c in enumerate(CLASS_NAMES):
    print(f"{i:<4} {c:<46} {tab.at[c, 'train']:>6} {tab.at[c, 'val']:>5} "
          f"{tab.at[c, 'test']:>5}")

xb, yb = next(iter(val_loader))
assert xb.shape[1:] == (3, IMG_SIZE, IMG_SIZE), f"unexpected batch shape {tuple(xb.shape)}"
assert int(yb.min()) >= 0 and int(yb.max()) < N_CLASSES, "labels out of range"
print(f"\nbatch {tuple(xb.shape)} {xb.dtype}, labels {tuple(yb.shape)}, "
      f"range [{xb.min():.2f}, {xb.max():.2f}]")

train 37997 | val 8149 | test 8136   batch 32, workers 0
imbalance 39x in train (Orange___Haunglongbing_(Citrus_greening) 3854 vs Potato___healthy 99)

idx  class                                           train   val  test
0    Apple___Apple_scab                                441    95    94
1    Apple___Black_rot                                 435    93    93
2    Apple___Cedar_apple_rust                          193    41    41
3    Apple___healthy                                  1027   306   305
4    Blueberry___healthy                               944   279   279
5    Cherry_(including_sour)___Powdery_mildew          736   158   158
6    Cherry_(including_sour)___healthy                 598   128   128
7    Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot    359    77    77
8    Corn_(maize)___Common_rust_                       834   179   179
9    Corn_(maize)___Northern_Leaf_Blight               689   148   148
10   Corn_(maize)___healthy                            814   17

---
---
# PART C - Train and evaluate

Transfer learning from ImageNet weights, with the classifier head swapped for one sized
to this dataset. `MODEL_KEY` in the config cell picks the architecture.

## B3. Model registry — one string instead of hand-written layer indices

Every backbone hides its classifier somewhere different: `fc`, `classifier`,
`classifier[1]`, `classifier[3]`, `classifier[6]`. Typing the wrong index is the most likely
way one of us silently trains a broken model, so the mapping lives here once, tested, and
`MODEL_KEY` picks from it. An unknown key raises with the full list of valid keys.

Weights come from `get_model(name, weights="DEFAULT")`, with the per-arch weights enum as a
fallback for older torchvision.

**SqueezeNet gets special treatment.** Its stock classifier ends
`Conv2d -> ReLU -> AdaptiveAvgPool2d`, so every logit is clamped to >= 0. A class the model
wants to *suppress* cannot go below zero, the softmax floor stays high, and the gradient that
would push it down is dead on arrival. We drop that ReLU:

`Dropout(0.5) -> Conv2d(512, 15, 1) -> AdaptiveAvgPool2d(1)`

`model.num_classes` is set too — SqueezeNet's `forward` flattens the classifier output, and
leaving the old value is a trap for anyone reading the object later.

In [11]:
# ---- LeafGuardNet: small backbone + CBAM, built for generalisation ---------- #
# Capacity did not help out of distribution (SqueezeNet 0.74M tied DenseNet 7.1M,
# while VGG16 134M came last), so this deliberately stays small and leans on the
# augmentation above plus attention rather than parameter count.
class ChannelAttention(nn.Module):
    """CBAM channel attention: which feature channels matter."""

    def __init__(self, ch, reduction=8):
        super().__init__()
        hidden = max(ch // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(ch, hidden), nn.ReLU(inplace=True),
                                 nn.Linear(hidden, ch))

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * torch.sigmoid(self.mlp(x.mean(dim=(2, 3)))
                                 + self.mlp(x.amax(dim=(2, 3)))).view(b, c, 1, 1)


class SpatialAttention(nn.Module):
    """CBAM spatial attention: WHERE the lesion is. This is the half a plain SE block
    cannot express, and lesion position/shape is what separates confusable diseases."""

    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        m = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.conv(m))


class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.channel, self.spatial = ChannelAttention(ch), SpatialAttention()

    def forward(self, x):
        return self.spatial(self.channel(x))


class LeafGuardNet(nn.Module):
    """MobileNetV3-Small features -> CBAM -> avg+max pooled head.

    Concatenating average and max pooling matters here: average pooling describes the
    leaf overall, max pooling fires on the strongest local response, which is what a
    small lesion on an otherwise healthy leaf looks like. Dropout 0.4 is deliberately
    heavy - the failure mode being fought is memorising a clean training domain.
    """

    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        w = torchvision.models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        self.features = torchvision.models.mobilenet_v3_small(weights=w).features
        ch = 576                                    # mobilenet_v3_small final width
        self.attn = CBAM(ch)
        self.classifier = nn.Sequential(
            nn.Linear(ch * 2, 512), nn.BatchNorm1d(512), nn.Hardswish(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, n_classes),
        )

    def forward(self, x):
        f = self.attn(self.features(x))
        pooled = torch.cat([f.mean(dim=(2, 3)), f.amax(dim=(2, 3))], dim=1)
        return self.classifier(pooled)


MODEL = LeafGuardNet(N_CLASSES).to(DEVICE)
PARAMS_TOTAL = sum(p.numel() for p in MODEL.parameters())
PARAMS_TRAINABLE = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
ARCH = "LeafGuardNet"

MODEL.eval()
with torch.no_grad():
    _out = MODEL(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
assert tuple(_out.shape) == (2, N_CLASSES), _out.shape
print(f"{MODEL_KEY}  ({ARCH})  head verified -> {tuple(_out.shape)}")
print(f"  parameters total     {PARAMS_TOTAL:>12,}")
print(f"  parameters trainable {PARAMS_TRAINABLE:>12,}")
print(f"  size at fp32         {PARAMS_TOTAL * 4 / 1e6:>12.2f} MB")

leafguard2  (LeafGuardNet)  head verified -> (2, 38)
  parameters total        1,621,553
  parameters trainable    1,621,553
  size at fp32                 6.49 MB


### Quote the parameter count for *this* dataset, not the ImageNet one

Every published SqueezeNet1.1 figure - 1.24 M - is measured at 1000 classes, and the
final `Conv2d(512, C, 1)` is a big slice of that: 513,000 weights at C=1000 against
19,494 at C=38. Swap the head and a large fraction of the model disappears. The count
printed above is the one that belongs in the report table.

The same correction applies to `alexnet` and `vgg16`, where `classifier[6]` is
`Linear(4096, C)`: 4.1 M weights at 1000 classes, 156 K at 38.

## B4. Loss, optimizer, scheduler

**Class weights are reordered, not trusted.** `class_weights.json` is keyed by class name;
`CLASS_NAMES` is alphabetical. Zipping the JSON's own order onto the label
indices would quietly train the model to weight the wrong classes — which looks like a bad
architecture, not a bug. Every name must be present or this cell raises.

Weights are `n_train / (15 * n_class)`, computed in Part A **from the train split only**.

The scheduler watches **val macro-F1**, not val loss. Under 21x imbalance a weighted loss and
macro-F1 can move in opposite directions, and macro-F1 is the number we are reporting.

In [12]:
CLASS_WEIGHTS = None
CW_SOURCE = "none (unweighted)"
cw_file = find_artifact("class_weights.json")

if not USE_CLASS_WEIGHTS:
    print("USE_CLASS_WEIGHTS = False -> unweighted loss")
elif cw_file is None:
    print("WARNING: class_weights.json not found in "
          + ", ".join(str(d) for d in ARTIFACT_DIRS))
    print("WARNING: falling back to an UNWEIGHTED loss. Rare classes will do worse and this")
    print("         run is not strictly comparable to runs that had the file. Copy it from")
    print("         notebook 01 and rerun if you can.")
else:
    raw = json.loads(Path(cw_file).read_text())
    missing = [c for c in CLASS_NAMES if c not in raw]
    assert not missing, (f"class_weights.json has no entry for {missing}. Keys present: "
                         f"{sorted(raw)}. Regenerate it with notebook 01.")
    extra = [k for k in raw if k not in CLASS_NAMES]
    if extra:
        print(f"note: ignoring {len(extra)} weight(s) for classes not on disk: {extra}")
    ordered = [float(raw[c]) for c in CLASS_NAMES]                 # <- the reordering
    CLASS_WEIGHTS = torch.tensor(ordered, dtype=torch.float32, device=DEVICE)
    CW_SOURCE = str(cw_file)
    print(f"class weights from {cw_file}, reordered to match CLASS_NAMES:")
    for i, c in enumerate(CLASS_NAMES):
        print(f"  {i:<3} {c:<46} {ordered[i]:.4f}")
    print(f"  ratio max/min {max(ordered) / min(ordered):.1f}x")

CRITERION = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS,
                                label_smoothing=LABEL_SMOOTHING)
OPTIMIZER = torch.optim.AdamW(MODEL.parameters(), lr=LR, weight_decay=1e-4)
SCHEDULER = ReduceLROnPlateau(OPTIMIZER, mode="max", factor=0.5, patience=3)

AMP_ON = bool(USE_AMP and DEVICE.type == "cuda")
AMP_DEVICE = "cuda" if DEVICE.type == "cuda" else "cpu"


def amp_autocast():
    """fp16 autocast on GPU, a clean no-op on CPU."""
    if AMP_ON:
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return torch.amp.autocast(AMP_DEVICE, enabled=False)


SCALER = torch.amp.GradScaler("cuda", enabled=AMP_ON)

METRICS = {
    "macro_precision": MulticlassPrecision(num_classes=N_CLASSES, average="macro").to(DEVICE),
    "macro_recall": MulticlassRecall(num_classes=N_CLASSES, average="macro").to(DEVICE),
    "macro_f1": MulticlassF1Score(num_classes=N_CLASSES, average="macro").to(DEVICE),
}

print(f"\nloss       CrossEntropyLoss(weight={'set' if CLASS_WEIGHTS is not None else 'None'})")
print(f"optimizer  Adam(lr={LR})")
print(f"scheduler  ReduceLROnPlateau(mode=max, factor=0.5, patience=3) on val macro-F1")
print(f"AMP        enabled={AMP_ON}  scaler={SCALER.is_enabled()}  device={AMP_DEVICE}")

class weights from c:\Users\Welcome\Desktop\npn\crop-disease-38\class_weights.json, reordered to match CLASS_NAMES:
  0   Apple___Apple_scab                             2.2674
  1   Apple___Black_rot                              2.2987
  2   Apple___Cedar_apple_rust                       5.1809
  3   Apple___healthy                                0.9736
  4   Blueberry___healthy                            1.0592
  5   Cherry_(including_sour)___Powdery_mildew       1.3586
  6   Cherry_(including_sour)___healthy              1.6721
  7   Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot 2.7853
  8   Corn_(maize)___Common_rust_                    1.1989
  9   Corn_(maize)___Northern_Leaf_Blight            1.4513
  10  Corn_(maize)___healthy                         1.2284
  11  Grape___Black_rot                              1.2106
  12  Grape___Esca_(Black_Measles)                   1.0330
  13  Grape___Leaf_blight_(Isariopsis_Leaf_Spot)     1.3262
  14  Grape___healthy                   

## B5. Train

`run_epoch` does both directions: pass an optimizer and it trains, leave it out and it
evaluates under `no_grad`. The scaler only ever touches the training path.

**The best checkpoint is chosen on val macro-F1.** On accuracy,
`Tomato__Tomato_YellowLeaf__Curl_Virus` at 481 val images outvotes `Potato___healthy` at 23
by 21:1, so an epoch that quietly stops predicting the rare class can still post the best
accuracy. Macro-F1 gives all 15 classes one vote each.

Per-epoch numbers stream to `log_<model>.csv` after every epoch, so a disconnected Colab
runtime still leaves the curve behind. Weights go to `best_<model>.pth`.

In [ ]:
LOG_PATH = OUT_DIR / f"log_{MODEL_KEY}.csv"
BEST_PATH = OUT_DIR / f"best_{MODEL_KEY}.pth"


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, desc=""):
    """One pass. Returns loss, accuracy, macro metrics, preds and targets.

    optimizer given -> training (grads on, scaler used). Omitted -> evaluation under no_grad.
    """
    training = optimizer is not None
    model.train(training)
    for m in METRICS.values():
        m.reset()                                  # stale state across epochs is silent
    tot_loss, n_seen, n_right = 0.0, 0, 0
    preds, targets = [], []

    for x, y in tqdm(loader, desc=desc, leave=False, unit="b"):
        x = x.to(DEVICE, non_blocking=PIN)
        y = y.to(DEVICE, non_blocking=PIN)
        with torch.set_grad_enabled(training):
            with amp_autocast():
                out = model(x)
                out = out if torch.is_tensor(out) else out[0]     # aux-head architectures
                loss = criterion(out, y)
        if training:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None and scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        p = out.detach().float().argmax(1)
        for m in METRICS.values():
            m.update(p, y)
        tot_loss += loss.item() * y.size(0)
        n_right += int((p == y).sum())
        n_seen += y.size(0)
        preds.append(p.cpu().numpy())
        targets.append(y.cpu().numpy())

    res = {k: float(m.compute()) for k, m in METRICS.items()}
    res.update(loss=tot_loss / max(n_seen, 1), accuracy=n_right / max(n_seen, 1),
               n=n_seen, preds=np.concatenate(preds), targets=np.concatenate(targets))
    return res


history, best = [], {"epoch": 0, "val_macro_f1": -1.0}
t_run = time.time()
print(f"{MODEL_KEY} for {EPOCHS} epochs on {DEVICE.type}, "
      f"selecting on val macro-F1\n" + "-" * 78)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    lr_used = OPTIMIZER.param_groups[0]["lr"]
    tr = run_epoch(MODEL, train_loader, CRITERION, OPTIMIZER, SCALER,
                   desc=f"{epoch}/{EPOCHS} train")
    va = run_epoch(MODEL, val_loader, CRITERION, desc=f"{epoch}/{EPOCHS} val")
    SCHEDULER.step(va["macro_f1"])
    secs = time.time() - t0

    history.append({
        "epoch": epoch, "lr": lr_used, "seconds": round(secs, 1),
        "train_loss": tr["loss"], "train_acc": tr["accuracy"],
        "train_macro_precision": tr["macro_precision"],
        "train_macro_recall": tr["macro_recall"], "train_macro_f1": tr["macro_f1"],
        "val_loss": va["loss"], "val_acc": va["accuracy"],
        "val_macro_precision": va["macro_precision"],
        "val_macro_recall": va["macro_recall"], "val_macro_f1": va["macro_f1"],
    })
    pd.DataFrame(history).to_csv(LOG_PATH, index=False)      # survives a lost runtime

    flag = ""
    if va["macro_f1"] > best["val_macro_f1"]:
        best = {"epoch": epoch, "val_macro_f1": va["macro_f1"], "val_acc": va["accuracy"]}
        torch.save(MODEL.state_dict(), BEST_PATH)
        flag = "  <- best"
    new_lr = OPTIMIZER.param_groups[0]["lr"]
    if new_lr < lr_used:
        flag += f"  lr {lr_used:.2e} -> {new_lr:.2e}"
    print(f"epoch {epoch:>2}/{EPOCHS}  {secs:>5.0f}s  "
          f"train loss {tr['loss']:.4f} acc {tr['accuracy']:.4f} f1 {tr['macro_f1']:.4f}  |  "
          f"val loss {va['loss']:.4f} acc {va['accuracy']:.4f} f1 {va['macro_f1']:.4f}{flag}")

TRAIN_MINUTES = (time.time() - t_run) / 60
print("-" * 78)
print(f"done in {TRAIN_MINUTES:.1f} min. best epoch {best['epoch']} "
      f"val macro-F1 {best['val_macro_f1']:.4f} (val acc {best['val_acc']:.4f})")
print(f"  {LOG_PATH}\n  {BEST_PATH}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
h = pd.DataFrame(history)
ax[0].plot(h["epoch"], h["train_loss"], label="train")
ax[0].plot(h["epoch"], h["val_loss"], label="val")
ax[0].set_title("loss")
ax[1].plot(h["epoch"], h["train_acc"], label="train")
ax[1].plot(h["epoch"], h["val_acc"], label="val")
ax[1].set_title("accuracy")
ax[2].plot(h["epoch"], h["train_macro_f1"], label="train")
ax[2].plot(h["epoch"], h["val_macro_f1"], label="val")
ax[2].axvline(best["epoch"], color="k", ls=":", lw=1, label=f"best (ep {best['epoch']})")
ax[2].set_title("macro-F1  <- selection metric")
for a in ax:
    a.set_xlabel("epoch")
    a.grid(alpha=0.3)
    a.legend()
fig.suptitle(f"{MODEL_KEY} - {DATA_SOURCE} split, seed {SEED}")
fig.tight_layout()
fig.savefig(OUT_DIR / f"curves_{MODEL_KEY}.png", dpi=110, bbox_inches="tight")
plt.show()

## B6. Evaluate once, reuse the predictions

The best checkpoint is loaded and each split is passed over exactly once. Everything below —
tables, reports, confusion matrices, image grids — reads from `preds_store`. Re-running the
model per figure would be slower and, with augmentation or dropout left on by accident, would
not even agree with itself.

`train` here is the `train_eval` loader, so the train/val gap is measured under identical
transforms.

**`Potato___healthy` has 23 val and 23 test images.** The 95% CI on its recall is about
±20 points, so one extra mistake moves it more than 4 points. Do not read a change in that
row as a real difference between two backbones without cross-validation.

In [ ]:
MODEL.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
print(f"loaded {BEST_PATH} (epoch {best['epoch']})\n")

preds_store = {}
for split, ld in (("train", train_eval_loader), ("val", val_loader), ("test", test_loader)):
    preds_store[split] = run_epoch(MODEL, ld, CRITERION, desc=f"eval {split}")
    r = preds_store[split]
    print(f"{split:<6} n={r['n']:<6} loss {r['loss']:.4f}  acc {r['accuracy']:.4f}  "
          f"macro-P {r['macro_precision']:.4f}  macro-R {r['macro_recall']:.4f}  "
          f"macro-F1 {r['macro_f1']:.4f}")

METRIC_TABLE = pd.DataFrame([{
    "model": MODEL_KEY, "split": s, "n": r["n"], "loss": round(r["loss"], 6),
    "accuracy": round(r["accuracy"], 6),
    "macro_precision": round(r["macro_precision"], 6),
    "macro_recall": round(r["macro_recall"], 6), "macro_f1": round(r["macro_f1"], 6),
    "data_source": DATA_SOURCE,
} for s, r in preds_store.items()])
METRIC_TABLE.to_csv(OUT_DIR / f"metrics_{MODEL_KEY}.csv", index=False)
print(f"\n{METRIC_TABLE.to_string(index=False)}")
gap = preds_store["train"]["macro_f1"] - preds_store["val"]["macro_f1"]
print(f"\ntrain-val macro-F1 gap {gap:+.4f}"
      + ("  (large gap = overfitting)" if gap > 0.05 else ""))

SHORT = [c.replace("___", " ").replace("__", " ").replace("_", " ")[:30] for c in CLASS_NAMES]


def wilson_ci(k, n, z=1.96):
    """95% Wilson score interval for a proportion.

    The normal (Wald) approximation is wrong exactly where it matters here: at
    recall 1.0 it returns a zero-width interval, claiming 33/33 proves the true
    recall is 100%. Wilson stays finite at the boundaries and is asymmetric, so
    a perfect small class reports something honest like [0.896, 1.000].
    """
    if not n:
        return float("nan"), float("nan")
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def per_class(split):
    """Per-class table plus a Wilson 95% CI on recall."""
    r = preds_store[split]
    rep = classification_report(r["targets"], r["preds"], labels=list(range(N_CLASSES)),
                                target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    rows = []
    for c in CLASS_NAMES:
        d = rep[c]
        n = int(d["support"])
        rec = d["recall"]
        lo, hi = wilson_ci(round(rec * n), n)
        rows.append({"model": MODEL_KEY, "split": split, "class": c, "support": n,
                     "precision": round(d["precision"], 6), "recall": round(rec, 6),
                     "f1": round(d["f1-score"], 6),
                     "recall_lo": round(lo, 4), "recall_hi": round(hi, 4),
                     "recall_ci95_width": round(hi - lo, 4)})
    return pd.DataFrame(rows)


for split in ("val", "test"):
    print("\n" + "=" * 78)
    print(f"classification report - {split}  ({MODEL_KEY}, {DATA_SOURCE})")
    print("=" * 78)
    print(classification_report(preds_store[split]["targets"], preds_store[split]["preds"],
                                labels=list(range(N_CLASSES)), target_names=CLASS_NAMES,
                                digits=4, zero_division=0))
    pc = per_class(split)
    pc.to_csv(OUT_DIR / f"per_class_{split}_{MODEL_KEY}.csv", index=False)
    worst = pc.sort_values("recall").head(3)
    print(f"weakest recall: " + ", ".join(
        f"{r['class']} {r['recall']:.3f} [{r['recall_lo']:.3f}, {r['recall_hi']:.3f}] "
        f"(n={r['support']})" for _i, r in worst.iterrows()))

thin = per_class("test").query("support < 30")
if len(thin):
    print("\nWARNING - too few test images for a stable per-class number:")
    for _i, r in thin.iterrows():
        print(f"  {r['class']:<46} n={r['support']:<4} recall {r['recall']:.3f} "
              f"95% CI [{r['recall_lo']:.3f}, {r['recall_hi']:.3f}]")
    print("  Quote the interval, or use k-fold CV. Do not call a change here an improvement.")

# ---------------------------- confusion matrices ---------------------------- #
CM = {s: confusion_matrix(preds_store[s]["targets"], preds_store[s]["preds"],
                          labels=list(range(N_CLASSES))) for s in ("val", "test")}


def draw_cm(ax, cm, title, normalize):
    data = cm.astype(float)
    if normalize:
        data = 100 * data / np.maximum(data.sum(axis=1, keepdims=True), 1)
    sns.heatmap(data, ax=ax, cmap="Blues", cbar=False, square=True,
                annot=True, fmt=".0f", annot_kws={"size": 7},
                xticklabels=SHORT, yticklabels=SHORT,
                vmin=0, vmax=100 if normalize else None)
    ax.set_title(title)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")


fig, axes = plt.subplots(1, 2, figsize=(26, 11))
for ax, s in zip(axes, ("val", "test")):
    draw_cm(ax, CM[s], f"{s} - counts (acc {preds_store[s]['accuracy']:.4f}, "
                       f"macro-F1 {preds_store[s]['macro_f1']:.4f})", normalize=False)
fig.suptitle(f"{MODEL_KEY} confusion matrices - {DATA_SOURCE} split", fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / f"confusion_val_test_{MODEL_KEY}.png", dpi=110, bbox_inches="tight")
plt.show()

for s in ("val", "test"):
    fig, ax = plt.subplots(figsize=(13, 11))
    draw_cm(ax, CM[s], f"{MODEL_KEY} - {s}, row-normalised % (diagonal = recall)",
            normalize=True)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"confusion_{s}_{MODEL_KEY}.png", dpi=110, bbox_inches="tight")
    plt.show()

off = CM["test"].copy()
np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1], off.shape))[0][:5]
print("worst test confusions (true -> predicted):")
for i, j in pairs:
    if off[i, j]:
        print(f"  {off[i, j]:>4}  {CLASS_NAMES[i]} -> {CLASS_NAMES[j]}")

# ------------------------------- image grids ------------------------------- #
paths = [p for p, _i in test_set.samples]
tp, pp = preds_store["test"]["targets"], preds_store["test"]["preds"]
assert len(paths) == len(tp), "test loader was shuffled, predictions are not aligned"


def grid(idx, title, fname):
    if len(idx) == 0:
        print(f"{title}: none")
        return
    idx = list(idx)[:16]
    side = int(np.ceil(len(idx) ** 0.5))
    fig, axes = plt.subplots(side, side, figsize=(3.1 * side, 3.4 * side))
    for ax, i in zip(np.atleast_1d(axes).ravel(), idx):
        ax.imshow(Image.open(paths[i]).convert("RGB"))
        good = tp[i] == pp[i]
        ax.set_title(f"true {SHORT[tp[i]]}\npred {SHORT[pp[i]]}", fontsize=8,
                     color="green" if good else "red")
        ax.axis("off")
    for ax in np.atleast_1d(axes).ravel()[len(idx):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=100, bbox_inches="tight")
    plt.show()


grid(RNG.choice(len(paths), size=min(16, len(paths)), replace=False),
     f"{MODEL_KEY} - 16 random test images", f"samples_{MODEL_KEY}.png")
wrong = np.flatnonzero(tp != pp)
grid(RNG.permutation(wrong), f"{MODEL_KEY} - misclassified ({len(wrong)} of {len(tp)})",
     f"errors_{MODEL_KEY}.png")

## B7. results_<model>.json — the file the team collects

Fixed schema, so the Part D table can read every run without special cases, and so
nobody has to retype numbers into a report table. It carries the settings as well as the scores: a result
whose provenance you cannot check is not a result.

`crop_disease_<model>.pth` is the deployable artefact — weights plus the class order and the
preprocessing they assume, because a state dict on its own is unusable six weeks later.

In [ ]:
RESULTS = {
    "model": MODEL_KEY,
    "arch": ARCH,
    "params_total": int(PARAMS_TOTAL),
    "params_trainable": int(PARAMS_TRAINABLE),
    "data_source": DATA_SOURCE,
    "data_root": str(SOURCE_DIR),
    "leakage_fixed": bool(LEAKAGE_FIXED),
    "seed": int(SEED),
    "epochs": int(EPOCHS),
    "lr": float(LR),
    "batch_size": int(BATCH_SIZE),
    "img_size": int(IMG_SIZE),
    "class_weighted": bool(CLASS_WEIGHTS is not None),
    "amp": bool(AMP_ON),
    "best_epoch": int(best["epoch"]),
    "best_val_macro_f1": round(float(best["val_macro_f1"]), 6),
    "n_classes": int(N_CLASSES),
    "class_names": CLASS_NAMES,
    "splits": {s: {
        "n": int(r["n"]),
        "loss": round(float(r["loss"]), 6),
        "accuracy": round(float(r["accuracy"]), 6),
        "macro_precision": round(float(r["macro_precision"]), 6),
        "macro_recall": round(float(r["macro_recall"]), 6),
        "macro_f1": round(float(r["macro_f1"]), 6),
    } for s, r in preds_store.items()},
    "per_class_test": {
        r["class"]: {"support": int(r["support"]), "precision": float(r["precision"]),
                     "recall": float(r["recall"]), "f1": float(r["f1"]),
                     "recall_lo": float(r["recall_lo"]),
                     "recall_hi": float(r["recall_hi"])}
        for _i, r in per_class("test").iterrows()},
    "train_minutes": round(float(TRAIN_MINUTES), 2),
    "device": (torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"),
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "class_weights_source": CW_SOURCE,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
}

res_path = OUT_DIR / f"results_{MODEL_KEY}.json"
res_path.write_text(json.dumps(RESULTS, indent=2))

ckpt_path = OUT_DIR / f"crop_disease_{MODEL_KEY}.pth"
torch.save({
    "state_dict": MODEL.state_dict(),
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "arch": ARCH,
    "model_key": MODEL_KEY,
    "normalize_mean": MEAN,
    "normalize_std": STD,
    "test_metrics": RESULTS["splits"]["test"],
}, ckpt_path)

print(json.dumps({k: v for k, v in RESULTS.items()
                  if k not in ("per_class_test", "class_names")}, indent=2))
print("\nfiles for the team folder:")
for f in sorted(OUT_DIR.glob(f"*{MODEL_KEY}*")):
    print(f"  {f.name:<44} {f.stat().st_size / 1024:>9.1f} KB")
print(f"\nresults_{MODEL_KEY}.json is the file to keep - the Part D table reads it.")

---
---
# PART D - Compare, and try your own photos

In [ ]:
# ---- every results_*.json in OUT_DIR, ranked by test macro-F1 --------------- #
found = []
for p in sorted(OUT_DIR.glob("*results*.json")):
    try:
        r = json.loads(Path(p).read_text())
    except Exception as e:
        print(f"skipping unreadable {p.name}: {e}")
        continue
    t = r["splits"]["test"]
    row = {"model": r.get("model", p.stem), "params": r.get("params_total", 0),
           "test_acc": round(t["accuracy"], 4), "test_macro_f1": round(t["macro_f1"], 4),
           "best_epoch": r.get("best_epoch"), "n_classes": r.get("n_classes")}
    if "test_tta" in r["splits"]:
        row["test_macro_f1_tta"] = round(r["splits"]["test_tta"]["macro_f1"], 4)
    found.append(row)

if not found:
    print(f"no results json found in {OUT_DIR}")
else:
    table = (pd.DataFrame(found).sort_values("test_macro_f1", ascending=False)
             .reset_index(drop=True))
    table["params"] = table["params"].map(lambda n: f"{n:,}")
    print(table.to_string(index=False))
    table.to_csv(OUT_DIR / "model_comparison.csv", index=False)
    print(f"\nsaved {OUT_DIR / 'model_comparison.csv'}")
    print("\nnote: rows come from every results json in this folder, including runs")
    print("from other notebooks. Recipes may differ (epochs, LR schedule, label")
    print("smoothing), so this ranks the builds as delivered - it is not a")
    print("controlled architecture-only comparison.")

---
## Part E - the number that actually matters: unseen images

`test_dataset/` holds web-sourced photos this model has never seen and which were not
drawn from PlantVillage. Scoring it here, in the same notebook, keeps the honest number
next to the flattering one.

The folders use the 15-class subset naming, so they are mapped onto the 38-class
training names explicitly - a silent mismatch would score every prediction as wrong.

In [ ]:
# ---- evaluate on the unseen, out-of-distribution set ----------------------- #
UNSEEN_DIR = Path("test_dataset")
NAME_MAP = {
    "Pepper__bell___Bacterial_spot": "Pepper,_bell___Bacterial_spot",
    "Pepper__bell___healthy": "Pepper,_bell___healthy",
    "Potato___Early_blight": "Potato___Early_blight",
    "Potato___Late_blight": "Potato___Late_blight",
    "Potato___healthy": "Potato___healthy",
    "Tomato_Bacterial_spot": "Tomato___Bacterial_spot",
    "Tomato_Early_blight": "Tomato___Early_blight",
    "Tomato_Late_blight": "Tomato___Late_blight",
    "Tomato_Leaf_Mold": "Tomato___Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato___Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite":
        "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato__Target_Spot": "Tomato___Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato__Tomato_mosaic_virus": "Tomato___Tomato_mosaic_virus",
    "Tomato_healthy": "Tomato___healthy",
}

if not UNSEEN_DIR.is_dir():
    print(f"{UNSEEN_DIR} not found - unzip test_dataset_imagefolder.zip to score it")
else:
    _items = []
    for _d in sorted(UNSEEN_DIR.iterdir()):
        if not _d.is_dir():
            continue
        _mapped = NAME_MAP.get(_d.name, _d.name)
        assert _mapped in CLASS_TO_IDX, f"unmapped folder {_d.name!r}"
        _items += [(f, _mapped) for f in sorted(_d.iterdir())
                   if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}]

    MODEL.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE, weights_only=True))
    MODEL.eval()
    _t1 = _t3 = _crop = 0
    _confs, _rows = [], []
    with torch.no_grad():
        for _f, _true in _items:
            with Image.open(_f) as _im:
                _x = eval_tf(_im.convert("RGB")).unsqueeze(0).to(DEVICE)
            _p = (MODEL(_x).softmax(1) + MODEL(torch.flip(_x, dims=[3])).softmax(1))[0]
            _p = (_p / _p.sum()).cpu().numpy()
            _order = np.argsort(_p)[::-1]
            _pred = CLASS_NAMES[_order[0]]
            _t1 += _pred == _true
            _t3 += _true in [CLASS_NAMES[i] for i in _order[:3]]
            _crop += _pred.split("___")[0] == _true.split("___")[0]
            _confs.append(float(_p[_order[0]]))
            _rows.append({"image": _f.name, "true": _true, "predicted": _pred,
                          "confidence": round(float(_p[_order[0]]), 4),
                          "correct": bool(_pred == _true)})

    _n = len(_items)
    print(f"UNSEEN set ({_n} images, flip-TTA):")
    print(f"  top-1 {_t1}/{_n} = {_t1/_n*100:.1f}%   top-3 {_t3}/{_n} = {_t3/_n*100:.1f}%")
    print(f"  crop identified {_crop}/{_n}   mean confidence {np.mean(_confs)*100:.1f}%")
    _t = preds_store["test"]["accuracy"] * 100
    print(f"\n  PlantVillage test {_t:.2f}%  ->  unseen {_t1/_n*100:.1f}%  "
          f"(gap {_t - _t1/_n*100:.1f} points)")
    pd.DataFrame(_rows).to_csv(OUT_DIR / f"unseen_{MODEL_KEY}.csv", index=False)
    print(f"  saved {OUT_DIR / f'unseen_{MODEL_KEY}.csv'}")
    print(pd.DataFrame(_rows)[["image", "true", "predicted", "confidence",
                               "correct"]].to_string(index=False))

## Try it on your own photo

Copy photos into the `my_photos/` folder next to this notebook and run the cell.
Predictions use flip-TTA - averaging the image with its mirror, which is free accuracy
because leaves have no canonical orientation.

**Confidence gate.** The model has no "none of these" option - softmax always sums to 1,
so it must name a class even for a crop it never saw. Anything below
`CONFIDENCE_THRESHOLD` is reported as UNCERTAIN rather than asserted as a diagnosis.
The threshold is calibrated from the model's own validation predictions, not chosen by
taste.

**What the gate cannot do.** It catches borderline cases, not confident nonsense. A
cross-entropy-trained backbone can rate an out-of-scope leaf at 100% - measured on this
project, SqueezeNet's confidence on out-of-scope photos sat *above* the 5th percentile
of its genuine predictions, so no threshold separates them. Label smoothing (used in
`lite_disease_net_v4_38.ipynb`) is what makes the gate actually work.

In [ ]:
CONFIDENCE_THRESHOLD = None    # None -> use the 5th percentile of correct val predictions

# ---- calibrate the gate on the validation split ---------------------------- #
MODEL.eval()
_vp, _vt = [], []
with torch.no_grad():
    for _x, _y in tqdm(val_loader, desc="calibrating", leave=False, unit="b"):
        _x = _x.to(DEVICE)
        _p = (MODEL(_x).softmax(1) + MODEL(torch.flip(_x, dims=[3])).softmax(1)) / 2
        _vp.append(_p.cpu())
        _vt.append(_y)
_vp, _vt = torch.cat(_vp).numpy(), torch.cat(_vt).numpy()
_ok = _vp.argmax(1) == _vt
if CONFIDENCE_THRESHOLD is None:
    CONFIDENCE_THRESHOLD = float(np.percentile(_vp.max(1)[_ok], 5))
print(f"gate = {CONFIDENCE_THRESHOLD:.3f}  "
      f"(5th percentile of confidence on correct val predictions; "
      f"keeps {(_vp.max(1) >= CONFIDENCE_THRESHOLD).mean():.1%} of val images)")

# ---- classify everything in my_photos/ ------------------------------------- #
PHOTO_DIR = Path("my_photos")
PHOTO_DIR.mkdir(exist_ok=True)
_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
_photos = sorted(p for p in PHOTO_DIR.iterdir()
                 if p.suffix.lower() in _exts and not p.name.startswith("prediction"))
if not _photos:
    print(f"\nNo photos found. Copy leaf photos into {PHOTO_DIR.resolve()} and rerun.")

_short = [c.replace("___", " ").replace("__", " ").replace("_", " ")[:30]
          for c in CLASS_NAMES]
for _f in _photos:
    try:
        _img = Image.open(_f).convert("RGB")
    except Exception as e:
        print(f"{_f.name}: not an image ({e})")
        continue
    _x = eval_tf(_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _prob = (MODEL(_x).softmax(1) + MODEL(torch.flip(_x, dims=[3])).softmax(1))[0]
        _prob = (_prob / _prob.sum()).cpu().numpy()
    _top = np.argsort(_prob)[::-1][:5]

    _fig, _ax = plt.subplots(1, 2, figsize=(12, 4.6),
                             gridspec_kw={"width_ratios": [1, 1.5]})
    _ax[0].imshow(_img)
    _ax[0].axis("off")
    _ax[0].set_title(_f.name[:40])
    _ax[1].barh([_short[i] for i in _top][::-1], _prob[_top][::-1],
                color=["#c7d9f1"] * 4 + ["#1f6fb4"])
    _ax[1].set_xlim(0, 1)
    _ax[1].set_xlabel("probability")
    _ax[1].set_title(f"{MODEL_KEY}+TTA: {CLASS_NAMES[_top[0]]} ({_prob[_top[0]]:.1%})")
    for _i, _v in enumerate(_prob[_top][::-1]):
        _ax[1].text(_v + 0.01, _i, f"{_v:.1%}", va="center", fontsize=9)
    _fig.tight_layout()
    plt.show()

    if _prob[_top[0]] >= CONFIDENCE_THRESHOLD:
        print(f"{_f.name} -> {CLASS_NAMES[_top[0]]}  {_prob[_top[0]]:.1%}")
    else:
        print(f"{_f.name} -> UNCERTAIN (best guess {CLASS_NAMES[_top[0]]} at "
              f"{_prob[_top[0]]:.1%}, below the {CONFIDENCE_THRESHOLD:.1%} gate)")
        print("  likely a crop or disease outside the trained classes, or a field "
              "photo unlike the studio training images")

## What goes in the report

- **Headline is macro-F1, not accuracy.** Under this imbalance a model can post high
  accuracy while missing most of the rare classes. Quote accuracy beside it, never alone.
- **Parameter counts are the ones printed in Part C**, measured with this
  dataset's head. Published ImageNet figures describe a different model.
- **State the split.** `leakage_fixed: true` means the train/eval overlap the raw
  folders contain is gone. Numbers from a naive split are inflated and belong in a
  different table.
- **Small classes carry wide error bars.** Any class with fewer than ~30 evaluation
  images has a recall confidence interval of roughly +/-20 points; differences smaller
  than that between two models are noise, not findings.
- **Confidence is not correctness.** A softmax score only ranks the classes the model
  knows. On a crop outside the training set it can be confidently wrong - which is what
  the gate in Part D, and label smoothing in the V4 notebook, are there to mitigate.
- **Lab data, not field data.** Uniform studio shots of single detached leaves. Expect
  a substantial drop on real field photography.